# 7 · One rule, many languages

**The problem.** Back to the four copies of case study 1. The QA team checks shipments in Excel; the data pipeline is
Python; the analysts use MATLAB; the specification is a LaTeX document. Each wants the rule in its own language, and
each copy, written by hand, drifts.

mbse-expressions models each of those languages as a **dialect**: its expressions are data too, it has its own
evaluator, and it writes its own source text. **Translators** between dialects turn the neutral rule (in the Basic
dialect, which the earlier case studies used) into each, and back.

In [1]:
import math

from mbse.Expressions import Evaluators, Expressions as E, Translators
from mbse.Expressions.Dialects import Excel, Latex, Matlab, Python
from mbse.Expressions.Dialects.Python import Text
from mbse.Expressions.Framework import Terms
from toolkit import cold_chain, show

store = cold_chain()
this, c = E.variable("this"), E.variable("c")
in_range = Text.FromFunction(lambda this: 2.0 <= this.celsius <= 8.0).data
allowed = this.readings.count_where("c", c.gt(8.0)).le(2).and_(this.readings.max().le(12.0)).data
languages = [(Excel.Expressions.DIALECT, Excel.Text), (Python.Expressions.DIALECT, Python.Text),
             (Matlab.Expressions.DIALECT, Matlab.Text), (Latex.Expressions.DIALECT, Latex.Text)]

## Step 1: translating

`Translators.between(source, target)` finds the translator, and `forward` translates. Each dialect's `Text.ToText`
writes its source:

In [2]:
for dialect, text in languages:
    print(f"{dialect.name():7}", text.ToText(Translators.between(E.DIALECT, dialect).forward(in_range)))

Excel   =AND(2.0 <= this.celsius, this.celsius <= 8.0)
Python  2.0 <= this.celsius and this.celsius <= 8.0
Matlab  2.0 <= this.celsius && this.celsius <= 8.0
Latex   2.0 \leq \mathit{this}.\mathit{celsius} \land \mathit{this}.\mathit{celsius} \leq 8.0


Collections translate to each language's idiom: `MAP` and `LAMBDA` in Excel, generator expressions in Python,
`arrayfun` in MATLAB. LaTeX has no counterpart for a quantifier, and says so:

In [3]:
for dialect, text in languages:
    try:
        print(f"{dialect.name():7}", text.ToText(Translators.between(E.DIALECT, dialect).forward(allowed)))
    except ValueError as error:
        print(f"{dialect.name():7}", error)

Excel   =AND(SUM(MAP(this.readings, LAMBDA(c, IF(c > 8.0, 1, 0)))) <= 2, MAX(this.readings) <= 12.0)
Python  sum(1 for c in this.readings if c > 8.0) <= 2 and max(this.readings) <= 12.0
Matlab  nnz(arrayfun(@(c) c > 8.0, this.readings)) <= 2 && max(this.readings) <= 12.0
Latex   Basic quantifier 'count' has no Latex counterpart


## Step 2: each language evaluates by its own rules

A translation keeps the rule's **form**. Its value is the target language's business, because that's where it will
run. Evaluate the translations on a shipment whose logger never reported:

In [4]:
silent = store.Shipment().id("S3").create()
excel = Translators.between(E.DIALECT, Excel.Expressions.DIALECT).forward(in_range)
python = Translators.between(E.DIALECT, Python.Expressions.DIALECT).forward(in_range)
print("Basic  ", Evaluators.OfAny(in_range, {"this": silent}))
print("Excel  ", Excel.Evaluators.OfAny(excel, {"this": silent}))
try:
    Python.Evaluators.OfAny(python, {"this": silent})
except AttributeError as error:
    print("Python ", type(error).__name__, error)

Basic   None
Excel   #FIELD!
Python  AttributeError 'Shipment' object has no attribute 'celsius'


Basic says unknown; Excel gives the error value `#FIELD!`, which propagates through a worksheet the way errors do
there; Python raises, as `shipment.celsius` would. Coercion differs too. Excel compares an integer limit with a float
reading happily, where Basic says unknown (case study 2):

In [5]:
sloppy = this.celsius.ge(2).data
cold = store.Shipment().id("S1").celsius(5.5).create()
print("Basic  ", Evaluators.OfAny(sloppy, {"this": cold}))
print("Excel  ", Excel.Evaluators.OfAny(Translators.between(E.DIALECT, Excel.Expressions.DIALECT).forward(sloppy), {"this": cold}))

Basic   None
Excel   True


So keep the rule in Basic, where its meaning is fixed, translate it to where it runs, and test it there with that
language's evaluator, which models its rules exactly.

## Step 3: reading rules back

A rule written in another language can be read into Basic, to analyze it (case study 6) or store it. In Python,
`Text.FromText` parses source into the Python dialect, and the translator goes the other way:

In [6]:
source = "2.0 <= this.celsius <= 8.0"
parsed = Python.Text.FromText(source)
back = Translators.between(Python.Expressions.DIALECT, E.DIALECT).forward(parsed)
print(show(back))
Terms.same(back, in_range)

and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0))


True

Round trips keep what both languages can say. What one of them can't, such as a let in MATLAB, which has none, is
written another way, and comes back in that other form.

## Step 4: what a rule may reach

A Python or MATLAB expression can call functions, and an expression loaded from a file mustn't be able to call
anything it likes. Each dialect's scope reaches only what the caller gives it: here, Python's `math`, and nothing
else:

In [7]:
rounded = Python.Text.FromText("import math\nmath.floor(this.celsius) <= 8")
print(Python.Text.ToText(rounded))
try:
    Python.Evaluators.OfAny(rounded, {"this": cold})
except ImportError as error:
    print(type(error).__name__, error)
Python.Evaluators.OfAny(rounded, Python.Evaluators.Scope({"this": cold}, modules={"math": math}))

import math
math.floor(this.celsius) <= 8
ImportError import of 'math' is not allowed by the scope


True

## Why it works this way

- **One translator per pair, by rules.** Each pair of dialects has its own translator, declared as rules pairing
  patterns, as the rewriting of case study 6 was. Translating directly or through Basic gives the same result.
- **Form, not value.** A translator can't make Excel treat a missing value as unknown, and shouldn't try: the
  translated rule must read the way an engineer in that language would write it.
- **Nothing is reachable by default.** Rules travel as data, sometimes from people you don't fully trust. A scope is an
  allowlist.